# 03 — Machine Learning

Two optional tasks are implemented: (1) rating regression, and (2) high-rating classification. The notebook only uses features that actually exist.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
sys.path.append('../src')
from utils import detect_columns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, classification_report, roc_auc_score

In [2]:
df=pd.read_csv('../outputs/movies_cleaned.csv')
cols=detect_columns(df)
for k in ['rating','popularity','votes','year']:
    c=cols.get(k)
    if c: df[c]=pd.to_numeric(df[c], errors='coerce')
cols

{'title': 'title',
 'rating': 'vote_average',
 'popularity': 'popularity',
 'votes': 'vote_count',
 'year': None,
 'release_date': 'release_date',
 'genre': 'genre',
 'overview': 'overview',
 'language': 'original_language'}

## A. Rating regression

In [3]:
target = cols.get("rating")
features = [cols.get(x) for x in ["popularity", "votes", "year"]] + ["genre_count"]
features = [c for c in features if c in df.columns and c != target]

data = df[features + [target]].dropna(subset=[target])
X, y = data[features], data[target]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42)

num = X.select_dtypes("number").columns
cat = X.select_dtypes(exclude="number").columns

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ]), cat)
])

model = Pipeline([
    ("pre", pre),
    ("rf", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))
])

model.fit(Xtr, ytr)
pred = model.predict(Xte)

print("MAE:", mean_absolute_error(yte, pred))
print("RMSE:", np.sqrt(mean_squared_error(yte, pred)))
print("R²:", r2_score(yte, pred))

MAE: 0.7033233638521533
RMSE: 0.8892459288514736
R²: 0.2748240146340437


## B. High-rating classification

In [4]:
if target and features:
    data = df[features + [target]].dropna(subset=[target]).copy()
    t = data[target].quantile(.80)
    data["high_rated"] = (data[target] >= t).astype(int)

    X, y = data[features], data["high_rated"]
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=.2, random_state=42, stratify=y
    )

    num = X.select_dtypes("number").columns
    cat = X.select_dtypes(exclude="number").columns

    pre = ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]), num),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]), cat)
    ])

    model = Pipeline([
        ("pre", pre),
        ("rf", RandomForestClassifier(
            n_estimators=300, random_state=42,
            class_weight="balanced", n_jobs=-1
        ))
    ])

    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    prob = model.predict_proba(Xte)[:, 1]

    print("Threshold:", round(t, 2))
    print("Accuracy :", round(accuracy_score(yte, pred), 4))
    print("ROC-AUC  :", round(roc_auc_score(yte, prob), 4))
    print(classification_report(yte, pred))

Threshold: 7.3
Accuracy : 0.7803
ROC-AUC  : 0.6
              precision    recall  f1-score   support

           0       0.81      0.94      0.87      1551
           1       0.45      0.19      0.26       415

    accuracy                           0.78      1966
   macro avg       0.63      0.56      0.57      1966
weighted avg       0.74      0.78      0.74      1966



### Modeling note

Avoid leakage: do not use rank fields that are directly computed from the target, or any feature that is derived from the rating you are trying to predict.